# Hybrid Video Game Recommender
----------------
**MSc Data Science Dissertation**  
**Author:** Vikrant Deshmukh  
**University:** University of Bristol  
**Project:** Video Game Recommendation System





## Notebook Objectives

This notebook combines candidate recommendations from three complementary models:

1. Metadata-based recommender  
2. Graph-based recommender  
3. Transformer semantic recommender  

The ranked candidate lists are combined using **Weighted Reciprocal Rank Fusion (RRF)**.

The notebook supports two uses:

- **Formal evaluation:** loads the saved Top-50 candidate files for the ten dissertation query games, produces Top-10 hybrid recommendations, records fusion runtime, and exports evaluation files.
- **Live recommendation:** loads the three source recommender notebooks and exposes `hybrid_recommend(game_title, top_n=15)` so any eligible catalogue game can be used as the query. This live interface is used by the separate hardware top-up notebook.

Rich-text TF-IDF is **not included** in the primary hybrid model.

In [ ]:
# importing Core libraries

import json
import time
import os
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

In [ ]:
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Dataset Loading and Configuration

The source recommenders have already generated their candidate files.  
This notebook loads those files directly instead of executing the three full model notebooks again.

In [80]:
# Configure the project and dataset paths.

if IN_COLAB:
    PROJECT_ROOT = Path("/content/drive/MyDrive/MSC_DISSERTATION")
else:
    PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "Data"
OUTPUT_DIR = PROJECT_ROOT / "Output/hybrid_outputs"

data_path = DATA_DIR / "games_clean_v1.csv"

if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found at: {data_path}")

games = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print("Dataset shape:", games.shape)

Dataset loaded successfully.
Dataset shape: (89618, 50)


## 2. Candidate File Configuration

Each source model must provide **50 candidates for each of the same 10 query games**.

The expected files are:

- `metadata_evaluation_candidates.csv`
- `graph_evaluation_candidates.csv`
- `semantic_evaluation_candidates.csv`

In [ ]:
# Main dissertation folders

PROJECT_DIR = Path(
    "/content/drive/MyDrive/MSC_DISSERTATION"
)

OUTPUT_DIR = PROJECT_DIR / "Output/evaluation_outputs"

# Fresh evaluation candidate files

CANDIDATE_PATHS = {
    "metadata": OUTPUT_DIR / "metadata_evaluation_candidates.csv",
    "graph": OUTPUT_DIR / "graph_evaluation_candidates.csv",
    "semantic": OUTPUT_DIR / "semantic_evaluation_candidates.csv",
}

# Check that every required file exists.

missing_files = []

for model_name in CANDIDATE_PATHS:

    file_path = CANDIDATE_PATHS[model_name]

    if not file_path.exists():

        missing_files.append(
            str(file_path)
        )

if len(missing_files) > 0:

    raise FileNotFoundError(
        "The following candidate files are missing:\n"
        + "\n".join(missing_files)
    )

print("All candidate files are available.")

All candidate files are available.


## 3. Load Candidate Recommendations

The three candidate files are loaded separately so their source ranks and similarity scores remain traceable.

In [ ]:
candidate_tables = {}

for model_name in CANDIDATE_PATHS:

    file_path = CANDIDATE_PATHS[model_name]

    candidate_table = pd.read_csv(
        file_path
    )

    candidate_tables[model_name] = (
        candidate_table
    )

    print(
        model_name.capitalize(),
        "shape:",
        candidate_table.shape
    )

Metadata shape: (500, 16)
Graph shape: (500, 16)
Semantic shape: (500, 13)


## 4. Validate Candidate Inputs

Before fusion, every source file is checked for:

- required columns;
- missing identifiers;
- duplicate query-candidate pairs;
- self-recommendations;
- exactly 50 candidates per query; and
- continuous ranks starting from 1.

In [ ]:
MODEL_COLUMNS = {
    "metadata": {
        "rank": "metadata_rank",
        "score": "metadata_score",
    },
    "graph": {
        "rank": "graph_rank",
        "score": "graph_score",
    },
    "semantic": {
        "rank": "semantic_rank",
        "score": "semantic_score",
    },
}

MODEL_ORDER = [
    "metadata",
    "graph",
    "semantic",
]

COMMON_COLUMNS = {
    "query_appid",
    "query_game",
    "appid",
    "recommended_game",
}

def validate_candidate_table(model_name, df, rank_col, score_col):
    required = {
        "query_appid", "query_game",
        "appid", "recommended_game",
        rank_col, score_col
    }

    if not required.issubset(df.columns):
        raise ValueError(f"{model_name}: missing required columns")

    if df[["query_appid", "appid"]].isna().any().any():
        raise ValueError(f"{model_name}: missing identifiers")

    if df.duplicated(["query_appid", "appid"]).any():
        raise ValueError(f"{model_name}: duplicate pairs")

    if (df["query_appid"] == df["appid"]).any():
        raise ValueError(f"{model_name}: self-recommendations")

    counts = df.groupby("query_appid").size()
    if not (counts == 50).all():
        raise ValueError(f"{model_name}: each query must have 50 candidates")

    for _, group in df.groupby("query_appid"):
        ranks = sorted(group[rank_col].astype(int))
        if ranks != list(range(1, len(group) + 1)):
            raise ValueError(f"{model_name}: rank issue")

In [ ]:
for model_name in MODEL_ORDER:
    validate_candidate_table(
        model_name,
        candidate_tables[model_name],
        MODEL_COLUMNS[model_name]["rank"],
        MODEL_COLUMNS[model_name]["score"],
    )

print("All candidate files passed validation.")

All candidate files passed validation.


In [ ]:
# Confirm that all models contain the same query App IDs.
EXPECTED_QUERY_COUNT = 10
shared_query_ids = None

for model_name in MODEL_ORDER:

    model_query_ids = set(
        candidate_tables[
            model_name
        ]["query_appid"].unique()
    )

    if shared_query_ids is None:

        shared_query_ids = model_query_ids

    else:

        shared_query_ids = (
            shared_query_ids.intersection(
                model_query_ids
            )
        )

shared_query_ids = sorted(
    shared_query_ids
)

if len(shared_query_ids) != EXPECTED_QUERY_COUNT:

    raise ValueError(
        "The models do not contain the same 10 query games."
    )

print(
    "Shared query games:",
    len(shared_query_ids)
)

Shared query games: 10


## 5. Hybrid Fusion Configuration

The hybrid recommender uses the final dissertation weights:

| Model | Weight |
|---|---:|
| Metadata | 0.40 |
| Graph | 0.40 |
| Semantic | 0.20 |

These weights were selected as design assumptions to balance the Metadata and Graph recommenders while retaining a smaller complementary Semantic contribution; their sensitivity is examined separately in the **evaluation notebook.**

For a candidate with rank \(r\), the model contribution is:
$$
\text{Weighted RRF contribution}
=
\frac{\text{model weight}}{60 + r}
$$

The final hybrid score is the sum of the three model contributions.

In [ ]:
MODEL_WEIGHTS = {
    "metadata": 0.40,
    "graph": 0.40,
    "semantic": 0.20,
}

RRF_CONSTANT = 60 # smoothing constant (standard baseline)
FINAL_TOP_N = 10

weight_total = sum(
    MODEL_WEIGHTS.values()
)

if not np.isclose(
    weight_total,
    1.0
):

    raise ValueError(
        "Model weights must sum to 1.0."
    )

print("Hybrid configuration is valid.")

Hybrid configuration is valid.


## 6. Prepare Candidate Tables

Only identifiers, titles, ranks and source-model scores are retained for fusion.

In [ ]:
fusion_tables = {}

for model_name in MODEL_ORDER:

    candidate_table = candidate_tables[
        model_name
    ]

    rank_column = MODEL_COLUMNS[
        model_name
    ]["rank"]

    score_column = MODEL_COLUMNS[
        model_name
    ]["score"]

    selected_columns = [
        "query_appid",
        "query_game",
        "appid",
        "recommended_game",
        rank_column,
        score_column,
    ]

    fusion_tables[model_name] = (
        candidate_table[
            selected_columns
        ].copy()
    )

    print(
        model_name.capitalize(),
        "fusion shape:",
        fusion_tables[
            model_name
        ].shape
    )

Metadata fusion shape: (500, 6)
Graph fusion shape: (500, 6)
Semantic fusion shape: (500, 6)


## 7. Weighted RRF Function

The function below merges the source candidate lists for one query game and
returns the final ranked Hybrid recommendations.

By default, the function uses the dissertation's selected Metadata 0.40,
Graph 0.40 and Semantic 0.20 weighting configuration.

An optional `model_weights` argument is also supported so that alternative
weight configurations can be evaluated later without changing the underlying
candidate lists or fusion procedure.

Source ranks and individual RRF contributions are retained for explainability.

In [ ]:
def fuse_candidate_lists(
    query_appid,
    top_n=10,
    source_tables=None,
    model_weights=None
):
    """
    Fuse ranked candidate tables using the dissertation's
    weighted Reciprocal Rank Fusion configuration.

    Parameters
    ----------
    query_appid : int
        App ID of the query game.

    top_n : int, default=10
        Number of final hybrid recommendations to return.

    source_tables : dict or None
        Candidate tables keyed by model name. If None,
        the saved evaluation fusion tables are used.
        Passing live candidate tables allows the same
        fusion logic to be reused by hybrid_recommend().
    """

    if top_n < 1:
        raise ValueError(
            "top_n must be at least 1."
        )

    if source_tables is None:
        source_tables = fusion_tables

    if model_weights is None:
      model_weights = MODEL_WEIGHTS

    weight_total = sum(
      model_weights.values()
    )

    if not np.isclose(
        weight_total,
        1.0
    ):
        raise ValueError(
            "Model weights must sum to 1.0."
        )

    model_query_tables = []
    available_models = []

    for model_name in MODEL_ORDER:

        if model_name not in source_tables:
            continue

        fusion_table = source_tables[
            model_name
        ]

        if fusion_table is None or fusion_table.empty:
            continue

        rank_column = MODEL_COLUMNS[
            model_name
        ]["rank"]

        score_column = MODEL_COLUMNS[
            model_name
        ]["score"]

        required_columns = {
            "query_appid",
            "query_game",
            "appid",
            "recommended_game",
            rank_column,
            score_column,
        }

        missing_columns = (
            required_columns.difference(
                set(fusion_table.columns)
            )
        )

        if missing_columns:
            raise ValueError(
                model_name
                + " candidate table is missing columns: "
                + str(
                    sorted(
                        missing_columns
                    )
                )
            )

        model_table = fusion_table[
            fusion_table["query_appid"]
            == query_appid
        ][
            [
                "query_appid",
                "query_game",
                "appid",
                "recommended_game",
                rank_column,
                score_column,
            ]
        ].copy()

        if model_table.empty:
            continue

        model_table = model_table.rename(
            columns={
                "query_game":
                    "query_game_"
                    + model_name,
                "recommended_game":
                    "recommended_game_"
                    + model_name,
            }
        )

        model_query_tables.append(
            model_table
        )

        available_models.append(
            model_name
        )

    if len(model_query_tables) == 0:
        return pd.DataFrame()

    merged_candidates = (
        model_query_tables[0]
    )

    table_position = 1

    while table_position < len(
        model_query_tables
    ):

        merged_candidates = (
            merged_candidates.merge(
                model_query_tables[
                    table_position
                ],
                on=[
                    "query_appid",
                    "appid",
                ],
                how="outer",
            )
        )

        table_position += 1

    merged_candidates[
        "query_game"
    ] = pd.Series(
        pd.NA,
        index=merged_candidates.index,
        dtype="object",
    )

    merged_candidates[
        "recommended_game"
    ] = pd.Series(
        pd.NA,
        index=merged_candidates.index,
        dtype="object",
    )

    for model_name in available_models:

        merged_candidates[
            "query_game"
        ] = merged_candidates[
            "query_game"
        ].combine_first(
            merged_candidates[
                "query_game_"
                + model_name
            ]
        )

        merged_candidates[
            "recommended_game"
        ] = merged_candidates[
            "recommended_game"
        ].combine_first(
            merged_candidates[
                "recommended_game_"
                + model_name
            ]
        )

    rrf_columns = []

    for model_name in MODEL_ORDER:

        rank_column = MODEL_COLUMNS[
            model_name
        ]["rank"]

        rrf_column = (
            model_name
            + "_rrf"
        )

        if model_name in available_models:

            merged_candidates[
                rrf_column
            ] = np.where(
                merged_candidates[
                    rank_column
                ].notna(),
                model_weights[
                    model_name
                ]
                / (
                    RRF_CONSTANT
                    + merged_candidates[
                        rank_column
                    ]
                ),
                0,
            )

        else:

            merged_candidates[
                rank_column
            ] = np.nan

            merged_candidates[
                rrf_column
            ] = 0.0

        rrf_columns.append(
            rrf_column
        )

    rank_columns = [
        MODEL_COLUMNS[
            model_name
        ]["rank"]
        for model_name in MODEL_ORDER
    ]

    merged_candidates[
        "model_agreement"
    ] = merged_candidates[
        rank_columns
    ].notna().sum(
        axis=1
    )

    merged_candidates[
        "hybrid_rrf_score"
    ] = merged_candidates[
        rrf_columns
    ].sum(
        axis=1
    )

    results = merged_candidates.sort_values(
        by=[
            "hybrid_rrf_score",
            "model_agreement",
        ],
        ascending=[
            False,
            False,
        ],
        kind="stable",
    ).reset_index(
        drop=True
    )

    results[
        "hybrid_rank"
    ] = np.arange(
        1,
        len(results) + 1,
    )

    output_columns = [
        "hybrid_rank",
        "query_appid",
        "query_game",
        "appid",
        "recommended_game",
        "metadata_rank",
        "graph_rank",
        "semantic_rank",
        "model_agreement",
        "metadata_rrf",
        "graph_rrf",
        "semantic_rrf",
        "hybrid_rrf_score",
    ]

    return results[
        output_columns
    ].head(
        top_n
    )


## 8. Generate Hybrid Recommendations

The function is applied to all ten shared query games.  
Runtime measures only the hybrid fusion stage, not the construction of the three source models.

In [ ]:
all_hybrid_results = []
hybrid_runtime_records = []

for query_appid in shared_query_ids:

    start_time = time.perf_counter()

    query_results = fuse_candidate_lists(
        query_appid=query_appid,
        top_n=FINAL_TOP_N,
    )

    end_time = time.perf_counter()

    if len(query_results) == 0:

        raise ValueError(
            "No hybrid results were generated for query App ID: "
            + str(query_appid)
        )

    query_runtime = (
        end_time - start_time
    )

    all_hybrid_results.append(
        query_results
    )

    query_name = query_results[
        "query_game"
    ].iloc[0]

    runtime_record = {
        "model": "Hybrid",
        "query_appid": query_appid,
        "query_game": query_name,
        "runtime_seconds": query_runtime,
    }

    hybrid_runtime_records.append(
        runtime_record
    )

    print(
        query_name,
        "| recommendations:",
        len(query_results),
        "| runtime:",
        round(
            query_runtime,
            6
        ),
        "seconds",
    )

hybrid_results = pd.concat(
    all_hybrid_results,
    ignore_index=True,
)

hybrid_runtime_results = pd.DataFrame(
    hybrid_runtime_records
)

print(
    "\nHybrid result shape:",
    hybrid_results.shape
)

print(
    "Runtime result shape:",
    hybrid_runtime_results.shape
)

Counter-Strike 2 | recommendations: 10 | runtime: 0.03959 seconds
Grand Theft Auto V Legacy | recommendations: 10 | runtime: 0.037163 seconds
Sid Meier’s Civilization® VI | recommendations: 10 | runtime: 0.032495 seconds
The Witcher 3: Wild Hunt | recommendations: 10 | runtime: 0.032207 seconds
Hollow Knight | recommendations: 10 | runtime: 0.031978 seconds
Stardew Valley | recommendations: 10 | runtime: 0.034655 seconds
Factorio | recommendations: 10 | runtime: 0.032002 seconds
Deep Rock Galactic | recommendations: 10 | runtime: 0.037778 seconds
Phasmophobia | recommendations: 10 | runtime: 0.035292 seconds
Hades | recommendations: 10 | runtime: 0.030298 seconds

Hybrid result shape: (100, 13)
Runtime result shape: (10, 4)


## 9. Validate Final Hybrid Output

The final output must contain:

- 10 query games;
- 10 recommendations per query;
- no duplicate query-candidate pairs;
- no self-recommendations; and
- continuous hybrid ranks from 1 to 10.

In [ ]:
hybrid_query_count = hybrid_results[
    "query_appid"
].nunique()

hybrid_candidate_counts = hybrid_results.groupby(
    "query_appid"
).size()

hybrid_duplicate_pairs = hybrid_results.duplicated(
    subset=[
        "query_appid",
        "appid",
    ]
).sum()

hybrid_self_recommendations = (
    hybrid_results["query_appid"]
    == hybrid_results["appid"]
).sum()

hybrid_rank_issue_count = 0

for query_appid, query_group in hybrid_results.groupby(
    "query_appid"
):

    actual_ranks = sorted(
        query_group["hybrid_rank"]
        .astype(int)
        .tolist()
    )

    expected_ranks = list(
        range(
            1,
            FINAL_TOP_N + 1,
        )
    )

    if actual_ranks != expected_ranks:

        hybrid_rank_issue_count = (
            hybrid_rank_issue_count + 1
        )

print(
    "Number of query games:",
    hybrid_query_count
)

print(
    "\nRecommendations per query:"
)

print(
    hybrid_candidate_counts
)

print(
    "\nDuplicate pairs:",
    hybrid_duplicate_pairs
)

print(
    "Self-recommendations:",
    hybrid_self_recommendations
)

print(
    "Queries with rank issues:",
    hybrid_rank_issue_count
)

if hybrid_query_count != EXPECTED_QUERY_COUNT:

    raise ValueError(
        "The hybrid output must contain 10 query games."
    )

if (
    hybrid_candidate_counts
    != FINAL_TOP_N
).any():

    raise ValueError(
        "Every query must contain exactly 10 hybrid recommendations."
    )

if hybrid_duplicate_pairs > 0:

    raise ValueError(
        "Duplicate hybrid recommendations were found."
    )

if hybrid_self_recommendations > 0:

    raise ValueError(
        "Self-recommendations were found."
    )

if hybrid_rank_issue_count > 0:

    raise ValueError(
        "Hybrid rank validation failed."
    )

print(
    "\nFinal hybrid output passed validation."
)

Number of query games: 10

Recommendations per query:
query_appid
730        10
271590     10
289070     10
292030     10
367520     10
413150     10
427520     10
548430     10
739630     10
1145360    10
dtype: int64

Duplicate pairs: 0
Self-recommendations: 0
Queries with rank issues: 0

Final hybrid output passed validation.


## 10. Inspect Results

The table below provides a compact view of the final rankings and source-model agreement.

In [ ]:
display_columns = [
    "query_game",
    "hybrid_rank",
    "recommended_game",
    "metadata_rank",
    "graph_rank",
    "semantic_rank",
    "model_agreement",
    "hybrid_rrf_score",
]

display(
    hybrid_results[
        display_columns
    ].head(20)
)

print(
    "\nModel agreement distribution:"
)

print(
    hybrid_results[
        "model_agreement"
    ].value_counts().sort_index()
)

,query_game,hybrid_rank,recommended_game,metadata_rank,graph_rank,semantic_rank,model_agreement,hybrid_rrf_score
0,Counter-Strike 2,1,Counter-Strike: Source,2.0,1.0,1.0,3,0.016288
1,Counter-Strike 2,2,Splitgate,1.0,5.0,NaN,2,0.012711
2,Counter-Strike 2,3,Tom Clancy's Rainbow Six® Siege,7.0,2.0,NaN,2,0.012422
3,Counter-Strike 2,4,Team Fortress 2,3.0,6.0,NaN,2,0.012410
4,Counter-Strike 2,5,Insurgency,6.0,3.0,NaN,2,0.012410
5,Counter-Strike 2,6,Ironsight,11.0,31.0,34.0,3,0.012157
6,Counter-Strike 2,7,VAIL VR,4.0,13.0,NaN,2,0.011729
7,Counter-Strike 2,8,Warfork,17.0,9.0,NaN,2,0.010992
8,Counter-Strike 2,9,Black Squad,5.0,28.0,NaN,2,0.010699
9,Counter-Strike 2,10,Lightphobe,41.0,4.0,NaN,2,0.010210



Model agreement distribution:
model_agreement
2    73
3    27
Name: count, dtype: int64


## 11. Export Evaluation Files

Two files are saved:

1. final Top-10 hybrid recommendations; and  
2. hybrid fusion runtime for each query game.

In [ ]:
HYBRID_OUTPUT_PATH = (
    OUTPUT_DIR
    / "hybrid_evaluation_recommendations.csv"
)

HYBRID_RUNTIME_PATH = (
    OUTPUT_DIR
    / "hybrid_evaluation_runtime.csv"
)

hybrid_results.to_csv(
    HYBRID_OUTPUT_PATH,
    index=False,
)

hybrid_runtime_results.to_csv(
    HYBRID_RUNTIME_PATH,
    index=False,
)

print(
    "Hybrid recommendations saved to:",
    HYBRID_OUTPUT_PATH
)

print(
    "Hybrid runtime saved to:",
    HYBRID_RUNTIME_PATH
)

Hybrid recommendations saved to: /content/drive/MyDrive/MSC_DISSERTATION/Output/evaluation_outputs/hybrid_evaluation_recommendations.csv
Hybrid runtime saved to: /content/drive/MyDrive/MSC_DISSERTATION/Output/evaluation_outputs/hybrid_evaluation_runtime.csv


## 12. RRF Weight Sensitivity Analysis

The selected hybrid recommender uses Metadata, Graph and Semantic weights of
0.40, 0.40 and 0.20 respectively. These weights were initially chosen as
design assumptions rather than learned parameters.

To assess whether the final hybrid ranking is sensitive to this choice,
alternative RRF weight configurations are tested using the same fixed source
candidate lists, the same ten formal evaluation queries and the same RRF
constant of \(k = 60\).

Four configurations are considered:

- **Current:** Metadata = 0.40, Graph = 0.40, Semantic = 0.20
- **Equal:** equal contribution from all three recommenders
- **Metadata-heavy:** increased Metadata contribution
- **Graph-heavy:** increased Graph contribution

All other parts of the hybrid fusion procedure remain unchanged.

In [ ]:
RRF_WEIGHT_CONFIGS = {

    "Current": {
        "metadata": 0.40,
        "graph": 0.40,
        "semantic": 0.20,
    },

    "Equal": {
        "metadata": 1 / 3,
        "graph": 1 / 3,
        "semantic": 1 / 3,
    },

    "Metadata-heavy": {
        "metadata": 0.50,
        "graph": 0.30,
        "semantic": 0.20,
    },

    "Graph-heavy": {
        "metadata": 0.30,
        "graph": 0.50,
        "semantic": 0.20,
    },
}


for config_name, weights in RRF_WEIGHT_CONFIGS.items():

    weight_total = sum(
        weights.values()
    )

    print(
        config_name,
        "|",
        weights,
        "| Total:",
        round(weight_total, 4),
    )

    if not np.isclose(
        weight_total,
        1.0,
    ):

        raise ValueError(
            config_name
            + " weights do not sum to 1.0."
        )

Current | {'metadata': 0.4, 'graph': 0.4, 'semantic': 0.2} | Total: 1.0
Equal | {'metadata': 0.3333333333333333, 'graph': 0.3333333333333333, 'semantic': 0.3333333333333333} | Total: 1.0
Metadata-heavy | {'metadata': 0.5, 'graph': 0.3, 'semantic': 0.2} | Total: 1.0
Graph-heavy | {'metadata': 0.3, 'graph': 0.5, 'semantic': 0.2} | Total: 1.0


### Generate Hybrid Recommendations for Each Weight Configuration

Each RRF weight configuration is now applied to the same ten formal evaluation
queries.

The underlying Metadata, Graph and Semantic Top-50 candidate lists remain
fixed. For every query, the existing `fuse_candidate_lists()` function is
reused with a different `model_weights` dictionary.

This produces a directly comparable Top-10 Hybrid recommendation set for each
weight configuration.

In [ ]:
sensitivity_results = []


for config_name, weights in RRF_WEIGHT_CONFIGS.items():

    print(
        "\nRunning configuration:",
        config_name
    )

    for query_appid in shared_query_ids:

        query_results = fuse_candidate_lists(
            query_appid=query_appid,
            top_n=FINAL_TOP_N,
            model_weights=weights,
        ).copy()

        query_results[
            "weight_configuration"
        ] = config_name

        query_results[
            "metadata_weight"
        ] = weights["metadata"]

        query_results[
            "graph_weight"
        ] = weights["graph"]

        query_results[
            "semantic_weight"
        ] = weights["semantic"]

        sensitivity_results.append(
            query_results
        )


rrf_sensitivity_results = pd.concat(
    sensitivity_results,
    ignore_index=True,
)


print(
    "\nSensitivity result shape:",
    rrf_sensitivity_results.shape
)

print(
    "\nRecommendations per configuration:"
)

print(
    rrf_sensitivity_results[
        "weight_configuration"
    ].value_counts()
)

display(
    rrf_sensitivity_results.sample(20)
)


Running configuration: Current

Running configuration: Equal

Running configuration: Metadata-heavy

Running configuration: Graph-heavy

Sensitivity result shape: (400, 17)

Recommendations per configuration:
weight_configuration
Current           100
Equal             100
Metadata-heavy    100
Graph-heavy       100
Name: count, dtype: int64


,hybrid_rank,query_appid,query_game,appid,recommended_game,metadata_rank,graph_rank,semantic_rank,model_agreement,metadata_rrf,graph_rrf,semantic_rrf,hybrid_rrf_score,weight_configuration,metadata_weight,graph_weight,semantic_weight
86,7,739630,Phasmophobia,3194340,We Are Alive,38.0,8.0,14.0,3,0.004082,0.005882,0.002703,0.012667,Current,0.400000,0.400000,0.200000
344,5,367520,Hollow Knight,597860,Nightmare Boy,5.0,7.0,NaN,2,0.004615,0.007463,0.000000,0.012078,Graph-heavy,0.300000,0.500000,0.200000
226,7,289070,Sid Meier’s Civilization® VI,65980,Sid Meier's Civilization®: Beyond Earth™,9.0,2.0,NaN,2,0.007246,0.004839,0.000000,0.012085,Metadata-heavy,0.500000,0.300000,0.200000
199,10,1145360,Hades,1037130,Dandy Ace,14.0,10.0,NaN,2,0.004505,0.004762,0.000000,0.009266,Equal,0.333333,0.333333,0.333333
35,6,292030,The Witcher 3: Wild Hunt,253980,Enclave,7.0,4.0,NaN,2,0.005970,0.006250,0.000000,0.012220,Current,0.400000,0.400000,0.200000
329,10,289070,Sid Meier’s Civilization® VI,226860,Galactic Civilizations III,19.0,32.0,29.0,3,0.003797,0.005435,0.002247,0.011479,Graph-heavy,0.300000,0.500000,0.200000
134,5,292030,The Witcher 3: Wild Hunt,606880,GreedFall,2.0,8.0,NaN,2,0.005376,0.004902,0.000000,0.010278,Equal,0.333333,0.333333,0.333333
359,10,413150,Stardew Valley,1707850,Cheaphaven,16.0,23.0,42.0,3,0.003947,0.006024,0.001961,0.011932,Graph-heavy,0.300000,0.500000,0.200000
98,9,1145360,Hades,524640,Asura: Vengeance Edition,16.0,7.0,NaN,2,0.005263,0.005970,0.000000,0.011233,Current,0.400000,0.400000,0.200000
356,7,413150,Stardew Valley,2627600,Project Real,5.0,5.0,NaN,2,0.004615,0.007692,0.000000,0.012308,Graph-heavy,0.300000,0.500000,0.200000


### Validate and Export the RRF Sensitivity Results

Before evaluating performance, the sensitivity output is validated to ensure
that every weight configuration contains the same ten evaluation queries and
exactly ten Hybrid recommendations per query.

The validated results are then exported so that the Evaluation notebook can
calculate Precision@10 and NDCG@10 for each RRF configuration.

In [ ]:
# ---------------------------------------------------------
# Validate sensitivity output
# ---------------------------------------------------------

configuration_count = (
    rrf_sensitivity_results[
        "weight_configuration"
    ].nunique()
)

query_count_by_config = (
    rrf_sensitivity_results
    .groupby(
        "weight_configuration"
    )["query_appid"]
    .nunique()
)

recommendation_counts = (
    rrf_sensitivity_results
    .groupby(
        [
            "weight_configuration",
            "query_appid",
        ]
    )
    .size()
)

duplicate_pairs = (
    rrf_sensitivity_results
    .duplicated(
        subset=[
            "weight_configuration",
            "query_appid",
            "appid",
        ]
    )
    .sum()
)

self_recommendations = (
    rrf_sensitivity_results[
        "query_appid"
    ]
    == rrf_sensitivity_results[
        "appid"
    ]
).sum()


print(
    "Configurations:",
    configuration_count
)

print(
    "\nQueries per configuration:"
)

print(
    query_count_by_config
)

print(
    "\nUnique recommendation counts:"
)

print(
    sorted(
        recommendation_counts.unique()
    )
)

print(
    "\nDuplicate pairs:",
    duplicate_pairs
)

print(
    "Self-recommendations:",
    self_recommendations
)


assert configuration_count == 4

assert (
    query_count_by_config
    == 10
).all()

assert (
    recommendation_counts
    == 10
).all()

assert duplicate_pairs == 0

assert self_recommendations == 0


print(
    "\nRRF sensitivity output passed validation."
)

Configurations: 4

Queries per configuration:
weight_configuration
Current           10
Equal             10
Graph-heavy       10
Metadata-heavy    10
Name: query_appid, dtype: int64

Unique recommendation counts:
[np.int64(10)]

Duplicate pairs: 0
Self-recommendations: 0

RRF sensitivity output passed validation.


In [ ]:
RRF_SENSITIVITY_PATH = (
    OUTPUT_DIR
    / "rrf_weight_sensitivity_results.csv"
)

rrf_sensitivity_results.to_csv(
    RRF_SENSITIVITY_PATH,
    index=False,
)

print(
    "RRF sensitivity results saved to:"
)

print(
    RRF_SENSITIVITY_PATH
)

RRF sensitivity results saved to:
/content/drive/MyDrive/MSC_DISSERTATION/Output/evaluation_outputs/rrf_weight_sensitivity_results.csv


## 13. Live Hybrid Recommendation Interface

The evaluation section above remains unchanged for reproducible dissertation results.

For the hardware top-up and later application interface, the cells below load the three source recommenders and expose a single live function:

`hybrid_recommend(game_title, top_n=15)`

The live function generates fresh Top-50 candidates from Metadata, Graph and Semantic models, then applies the **same 0.40 / 0.40 / 0.20 weighted RRF logic** used in formal evaluation.

A source model may occasionally be unable to represent a particular game because of missing metadata or descriptive text. The live wrapper continues with the available source models rather than failing immediately. The original model weights are not renormalised.


In [ ]:
# Paths to the three source recommender notebooks.

LIVE_NOTEBOOK_DIR = (
    PROJECT_DIR
    / "Notebooks"
    / "Final Notebooks"
)

LIVE_SOURCE_NOTEBOOKS = {
    "semantic": (
        LIVE_NOTEBOOK_DIR
        / "03_transformer_semantic_recommender.ipynb"
    ),
    "graph": (
        LIVE_NOTEBOOK_DIR
        / "04_graph_based_recommender_final.ipynb"
    ),
    "metadata": (
        LIVE_NOTEBOOK_DIR
        / "01_popularity_and_weighted_metadata_recommender_final.ipynb"
    ),
}

missing_live_notebooks = [
    str(path)
    for path in LIVE_SOURCE_NOTEBOOKS.values()
    if not path.exists()
]

if missing_live_notebooks:
    raise FileNotFoundError(
        "Missing live recommender notebook(s):\n"
        + "\n".join(
            missing_live_notebooks
        )
    )

print(
    "Live source notebooks are available."
)


Live source notebooks are available.


In [ ]:
# Load source models only when their live model state
# is not already available in the current session.
#
# Order matters: Metadata is checked last because its
# recommender uses the shared global `games` DataFrame.
# Graph and Semantic use model-specific DataFrames.

from IPython.utils.capture import capture_output


def _live_source_ready(
    model_name
):
    if model_name == "semantic":

        return (
            callable(
                globals().get(
                    "generate_semantic_candidates"
                )
            )
            and "semantic_games" in globals()
            and "semantic_embeddings" in globals()
            and "semantic_title_to_model_row" in globals()
        )

    if model_name == "graph":

        return (
            callable(
                globals().get(
                    "generate_graph_candidates"
                )
            )
            and "graph_games" in globals()
            and "graph_title_to_game_index" in globals()
            and "normalised_game_feature_matrix" in globals()
        )

    if model_name == "metadata":

        metadata_columns = {
            "display_name",
            "genres_list",
            "tags_list",
            "play_mode_features",
        }

        return (
            callable(
                globals().get(
                    "generate_metadata_candidates"
                )
            )
            and "games" in globals()
            and metadata_columns.issubset(
                set(
                    games.columns
                )
            )
            and "metadata_nn" in globals()
            and "title_to_game_position" in globals()
            and "eligible_core_metadata_matrix" in globals()
        )

    return False


for model_name in [
    "semantic",
    "graph",
    "metadata",
]:

    if not _live_source_ready(
        model_name
    ):

        notebook_path = (
            LIVE_SOURCE_NOTEBOOKS[
                model_name
            ]
        )

        with capture_output():
            get_ipython().run_line_magic(
                "run",
                f'"{notebook_path}"',
            )


for model_name in [
    "semantic",
    "graph",
    "metadata",
]:

    if not _live_source_ready(
        model_name
    ):

        raise RuntimeError(
            f"{model_name.capitalize()} live "
            "recommender did not load correctly."
        )


print(
    "Live source recommenders loaded: "
    "semantic, graph, metadata"
)


Live source recommenders loaded: semantic, graph, metadata


In [ ]:
LIVE_CANDIDATE_TOP_N = 50
LIVE_CANDIDATE_POOL = 100
LIVE_DEFAULT_TOP_N = 15


def hybrid_recommend(
    game_title,
    top_n=LIVE_DEFAULT_TOP_N,
):
    """
    Generate live hybrid recommendations for a game title.

    Fresh candidate lists are generated by the Metadata,
    Graph and Semantic recommenders, then fused with the
    same weighted RRF configuration used in evaluation.
    """

    if top_n < 1:
        raise ValueError(
            "top_n must be at least 1."
        )

    candidate_top_n = max(
        LIVE_CANDIDATE_TOP_N,
        top_n,
    )

    candidate_pool = max(
        LIVE_CANDIDATE_POOL,
        candidate_top_n,
    )

    generators = {
        "metadata":
            generate_metadata_candidates,
        "graph":
            generate_graph_candidates,
        "semantic":
            generate_semantic_candidates,
    }

    live_candidate_tables = {}
    model_errors = {}

    for model_name in MODEL_ORDER:

        generator = generators[
            model_name
        ]

        try:

            model_results = generator(
                query_title=game_title,
                top_n=candidate_top_n,
                candidate_pool=candidate_pool,
            )

            if (
                model_results is not None
                and not model_results.empty
            ):

                live_candidate_tables[
                    model_name
                ] = model_results

        except (
            ValueError,
            KeyError,
            IndexError,
        ) as error:

            model_errors[
                model_name
            ] = str(
                error
            )

    if len(live_candidate_tables) == 0:

        error_text = (
            "; ".join(
                f"{model}: {message}"
                for model, message
                in model_errors.items()
            )
        )

        raise ValueError(
            "No source recommender could generate "
            f"candidates for '{game_title}'. "
            + error_text
        )

    query_appids = {
        int(
            table["query_appid"].iloc[0]
        )
        for table
        in live_candidate_tables.values()
    }

    if len(query_appids) != 1:

        raise ValueError(
            "Source recommenders resolved the query "
            "title to different App IDs."
        )

    query_appid = next(
        iter(
            query_appids
        )
    )

    results = fuse_candidate_lists(
        query_appid=query_appid,
        top_n=top_n,
        source_tables=live_candidate_tables,
    )

    if results.empty:

        raise ValueError(
            "No hybrid recommendations were generated "
            f"for '{game_title}'."
        )

    results.attrs[
        "source_model_errors"
    ] = model_errors

    return results


### Live usage

The hardware top-up notebook can now load this notebook and call:

```python
results = hybrid_recommend(
    "Elden Ring",
    top_n=15,
)
```

The returned table retains `appid`, `recommended_game`, `hybrid_rank`, `hybrid_rrf_score`, and `model_agreement`, so the hardware layer can directly use the recommended App IDs to fetch Steam PC requirements.


In [ ]:
# Small smoke test for the live interface.
# Change the title to any eligible catalogue game.

live_test_results = hybrid_recommend(
    "Raji: An Ancient Epic",
    top_n=15,
)

display(
    live_test_results[
        [
            "hybrid_rank",
            "appid",
            "recommended_game",
            "hybrid_rrf_score",
            "model_agreement",
        ]
    ]
)


,hybrid_rank,appid,recommended_game,hybrid_rrf_score,model_agreement
0,1,2653880,Amber,0.013115,2
1,2,2513170,Origami Lovers,0.012512,2
2,3,2735110,An Eternity Gone By,0.012503,2
3,4,916140,The Tale of Bistun,0.012220,2
4,5,2021040,Shards of Nogard,0.012007,2
5,6,645320,SCARF,0.011120,2
6,7,1015890,TASOMACHI: Behind the Twilight,0.011120,2
7,8,2169570,The Lost Legends of Redwall™: The Scout Anthology,0.011069,2
8,9,843180,Clan O'Conall and the Crown of the Stag,0.010797,2
9,10,1977170,Jusant,0.010658,2


## Conclusion

The notebook now supports both dissertation evaluation and live inference.

- The existing saved-candidate pipeline preserves reproducible Top-10 evaluation results for the ten formal query games.
- `hybrid_recommend(game_title, top_n=15)` provides live recommendations for downstream use.
- Both modes use the same Metadata 0.40, Graph 0.40 and Semantic 0.20 weighted RRF configuration.
- The live output is directly compatible with the separate hardware top-up notebook because each recommendation retains its Steam `appid`.


In [90]:
live_test_results = hybrid_recommend(
    "call of duty",
    top_n=15,
)

live_test_results


,hybrid_rank,query_appid,query_game,appid,recommended_game,metadata_rank,graph_rank,semantic_rank,model_agreement,metadata_rrf,graph_rrf,semantic_rrf,hybrid_rrf_score
0,1,1938090,Call of Duty®,1962660,Call of Duty®: Modern Warfare® II,NaN,1.0,6.0,2,0.0,0.006557,0.003030,0.009588
1,2,1938090,Call of Duty®,2519060,Call of Duty®: Modern Warfare® III,NaN,2.0,9.0,2,0.0,0.006452,0.002899,0.009350
2,3,1938090,Call of Duty®,1985810,Call of Duty®: Black Ops Cold War,NaN,5.0,3.0,2,0.0,0.006154,0.003175,0.009328
3,4,1938090,Call of Duty®,2933620,Call of Duty®: Black Ops 6,NaN,3.0,11.0,2,0.0,0.006349,0.002817,0.009166
4,5,1938090,Call of Duty®,2000950,Call of Duty®: Modern Warfare®,NaN,4.0,17.0,2,0.0,0.006250,0.002597,0.008847
5,6,1938090,Call of Duty®,1985820,Call of Duty®: Vanguard,NaN,6.0,12.0,2,0.0,0.006061,0.002778,0.008838
6,7,1938090,Call of Duty®,393080,Call of Duty®: Modern Warfare® Remastered (2017),NaN,12.0,10.0,2,0.0,0.005556,0.002857,0.008413
7,8,1938090,Call of Duty®,302670,Call to Arms,NaN,20.0,35.0,2,0.0,0.005000,0.002105,0.007105
8,9,1938090,Call of Duty®,581320,Insurgency: Sandstorm,NaN,7.0,NaN,1,0.0,0.005970,0.000000,0.005970
9,10,1938090,Call of Duty®,2659690,Operation Athena,NaN,8.0,NaN,1,0.0,0.005882,0.000000,0.005882
